# Fine-tuned Transformer vs Gemini

Side-by-side comparison of your fine-tuned T5 against Google Gemini 2.5 Flash.

**Setup:**
- Create `.env` in the project root with `GEMINI_API_KEY=...`
- The next cell auto-loads `.env`
- Choose the fine-tuned model via `FT_MODEL_NAME` (default: `t5-small-paraphrase`)

In [1]:
# === Configuration ===
GEMINI_MODEL     = 'gemini-2.5-flash'

# Which fine-tuned model to compare against Gemini.
# Change to 'pegasus-paraphrase' to swap.
FT_MODEL_NAME    = 't5-small-paraphrase'

N_PARAPHRASES    = 3

# Rough USD/1M-token prices (update if stale)
LLM_PRICE_PER_MTOK = {
    'gemini-2.5-flash':    {'input': 0.10, 'output':  0.40},
}

In [2]:
import os, sys, time, json, re
from pathlib import Path

# Auto-load .env at the project root for the Gemini API key
try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / '.env' if Path.cwd().name == 'notebooks' else '.env')
except ImportError:
    print('Note: pip install python-dotenv to use .env files')

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FT_MODEL_PATH = REPO / 'checkpoints' / FT_MODEL_NAME

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Fine-tuned model path:', FT_MODEL_PATH, '(exists:', FT_MODEL_PATH.exists(), ')')
print('LLM: gemini /', GEMINI_MODEL)

Device: mps
Fine-tuned model path: /Users/kunal/Documents/telus_assessment/checkpoints/t5-small-paraphrase (exists: True )
LLM: gemini / gemini-2.5-flash


In [ ]:
# API key sanity check
if not os.environ.get('GEMINI_API_KEY'):
    print('WARNING: GEMINI_API_KEY is not set. Add it to .env or export it, then restart the kernel.')
else:
    print('GEMINI_API_KEY: set')

GEMINI_API_KEY: set


In [4]:
# Load the fine-tuned model
ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_PATH)
ft_model     = AutoModelForSeq2SeqLM.from_pretrained(FT_MODEL_PATH).to(DEVICE)
ft_model.eval()
FT_IS_T5 = ft_model.config.model_type == 't5'
print(f'Loaded {FT_MODEL_NAME}  (model_type={ft_model.config.model_type})')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loaded t5-small-paraphrase  (model_type=t5)


In [5]:
@torch.no_grad()
def finetuned_paraphrase(text, n=N_PARAPHRASES, num_beams=5, max_length=128):
    prefix = 'paraphrase: ' if FT_IS_T5 else ''
    inputs = ft_tokenizer(prefix + text, return_tensors='pt',
                          truncation=True, max_length=max_length).to(DEVICE)
    outputs = ft_model.generate(
        **inputs,
        max_length=max_length,
        num_beams=max(num_beams, n),
        num_return_sequences=n,
    )
    return [ft_tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [6]:
LLM_PROMPT_TEMPLATE = (
    'Paraphrase the following sentence in {n} different ways. '
    'Each paraphrase should preserve the meaning but use different wording and structure. '
    'Return ONLY a JSON array of {n} strings, no commentary, no markdown fences.\n\n'
    'Sentence: {text}'
)

def _parse_llm_output(raw, n):
    """Try JSON parse; fall back to splitting by newlines."""
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip(), flags=re.MULTILINE)
    try:
        arr = json.loads(cleaned)
        if isinstance(arr, list):
            return [str(x) for x in arr[:n]]
    except json.JSONDecodeError:
        pass
    lines = [re.sub(r'^[\d\.\-\*\s]+', '', l).strip() for l in raw.splitlines() if l.strip()]
    return lines[:n]

from google import genai
_client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

def llm_paraphrase(text, n=N_PARAPHRASES):
    resp = _client.models.generate_content(
        model=GEMINI_MODEL,
        contents=LLM_PROMPT_TEMPLATE.format(n=n, text=text),
    )
    raw = resp.text or ''
    u = getattr(resp, 'usage_metadata', None)
    in_tok  = getattr(u, 'prompt_token_count', 0) if u else 0
    out_tok = getattr(u, 'candidates_token_count', 0) if u else 0
    return _parse_llm_output(raw, n), (in_tok, out_tok)

print(f'LLM client ready: gemini / {GEMINI_MODEL}')

LLM client ready: gemini / gemini-2.5-flash


In [7]:
PROMPTS = [
    'The quick brown fox jumps over the lazy dog.',
    'I forgot my password and the recovery email is no longer active.',
    'Climate change is one of the most pressing challenges of our time.',
    'Machine learning models require large amounts of training data to perform well.',
    'She walked into the room and immediately noticed that something was different.',
    'The stock market experienced a sharp decline due to rising interest rates.',
    'Could you tell me how to get to the train station from here?',
    'Quantum computers use qubits which can exist in superposition of states.',
]
print(f'{len(PROMPTS)} prompts')

8 prompts


In [8]:
rows = []
ft_latencies, llm_latencies = [], []
total_in_tok, total_out_tok = 0, 0

for src in tqdm(PROMPTS):
    t0 = time.perf_counter()
    ft_outs = finetuned_paraphrase(src)
    ft_latencies.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    try:
        llm_outs, (in_tok, out_tok) = llm_paraphrase(src)
        total_in_tok  += in_tok
        total_out_tok += out_tok
    except Exception as e:
        llm_outs = [f'<API error: {e}>'] * N_PARAPHRASES
    llm_latencies.append(time.perf_counter() - t0)

    row = {'source': src}
    for i, o in enumerate(ft_outs, 1):    row[f'FT_{i}']     = o
    for i, o in enumerate(llm_outs, 1):   row[f'Gemini_{i}'] = o
    rows.append(row)

df = pd.DataFrame(rows)
df

  0%|          | 0/8 [00:00<?, ?it/s]

,source,FT_1,FT_2,FT_3,Gemini_1,Gemini_2,Gemini_3
0,The quick brown fox jumps over the lazy dog.,The quick brown fox jumps over the lazy dog.,The quick brown fox leaps over the lazy dog.,The lazy dog is hit by a quick brown fox.,"A swift, russet fox leaps across the idle canine.",The agile auburn fox clears the sluggish dog.,"Making a quick leap, the brown fox goes over t..."
1,I forgot my password and the recovery email is...,I forgot my password and the recovery email is...,My password was forgotten and the recovery ema...,The recovery email is no longer active and I f...,I'm unable to access my account because I've f...,"My password has slipped my mind, and to make m...",The email address set up for regaining access ...
2,Climate change is one of the most pressing cha...,Climate change is one of the most pressing cha...,One of the most pressing challenges of our tim...,Climate change is one of the pressing challeng...,Climate change stands as one of the most param...,Among the critical challenges confronting the ...,The contemporary era faces few issues as signi...
3,Machine learning models require large amounts ...,The performance of machine learning models req...,Machine learning models require extensive trai...,"In machine learning models, training data is r...","To achieve high performance, machine learning ...",Optimal functioning of machine learning models...,Machine learning models only perform effective...
4,She walked into the room and immediately notic...,She walked into the room and immediately notic...,She walked into the room and immediately reali...,She walked into the room and immediately disco...,"The instant she entered the room, she perceive...","Upon her entry into the room, it was immediate...","She stepped into the room, and without delay, ..."
5,The stock market experienced a sharp decline d...,The stock market experienced a significant dec...,The stock market experienced a sharp decline d...,The stock market experienced a significant dec...,Rising interest rates caused a steep drop in t...,A significant downturn in the stock market occ...,The stock market fell sharply as interest rate...
6,Could you tell me how to get to the train stat...,Can you provide me with instructions on how to...,What are the steps to reach the train station ...,Can you explain how to reach the train station...,<API error: 429 RESOURCE_EXHAUSTED. {'error': ...,<API error: 429 RESOURCE_EXHAUSTED. {'error': ...,<API error: 429 RESOURCE_EXHAUSTED. {'error': ...
7,Quantum computers use qubits which can exist i...,Quantum computers utilize qubits that can exis...,Quantum computers utilize qubits that can be u...,Quantum computers utilize qubits that can be u...,<API error: 429 RESOURCE_EXHAUSTED. {'error': ...,<API error: 429 RESOURCE_EXHAUSTED. {'error': ...,<API error: 429 RESOURCE_EXHAUSTED. {'error': ...


In [9]:
# Latency + cost summary
ft_avg  = sum(ft_latencies)  / len(ft_latencies)
llm_avg = sum(llm_latencies) / len(llm_latencies)
print(f'Fine-tuned avg latency: {ft_avg*1000:>6.0f} ms/prompt')
print(f'Gemini     avg latency: {llm_avg*1000:>6.0f} ms/prompt  ({llm_avg/ft_avg:.1f}x slower)' )

price = LLM_PRICE_PER_MTOK.get(GEMINI_MODEL)
if price and total_in_tok:
    cost = (total_in_tok / 1e6) * price['input'] + (total_out_tok / 1e6) * price['output']
    per_call = cost / len(PROMPTS)
    print(f'\nTokens: in={total_in_tok}  out={total_out_tok}')
    print(f'Gemini cost: ${cost:.5f} total  (~${per_call:.5f}/prompt, ~${per_call*1000:.2f}/1k prompts)')
    print('Fine-tuned cost: $0 (local inference) plus electricity / amortized training time')

Fine-tuned avg latency:    350 ms/prompt
Gemini     avg latency:   5192 ms/prompt  (14.8x slower)

Tokens: in=351  out=352
Gemini cost: $0.00018 total  (~$0.00002/prompt, ~$0.02/1k prompts)
Fine-tuned cost: $0 (local inference) plus electricity / amortized training time


In [10]:
# Save results next to the notebook for the writeup
out_csv = Path(f'comparison_{FT_MODEL_NAME}_vs_gemini.csv')
df.to_csv(out_csv, index=False)
print(f'Wrote {out_csv}')

Wrote comparison_t5-small-paraphrase_vs_gemini.csv


## Notes & analysis

_(Fill in after running the cells.)_

**Quality:**
- Where does Gemini clearly win? (Long prompts, technical jargon, unusual sentence structures?)
- Where is fine-tuned competitive? (Short, common sentence structures it saw a lot of during training?)
- Any cases where the fine-tuned model produced more diverse paraphrases than Gemini?

**Trade-offs:**
- Latency: fine-tuned is ~Nx faster locally
- Cost at scale: ~$X per 1k prompts for Gemini vs $0 marginal for the fine-tuned model
- Operational: Gemini has zero training cost but requires network + API key; fine-tuned ships as a self-contained checkpoint

**When to pick which:**
- Fine-tuned: high volume, latency-sensitive, offline / privacy-sensitive workloads
- Gemini: low volume, quality-critical, need to handle out-of-distribution inputs